In [33]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# DEPENDENCIES

In [34]:
from bs4 import BeautifulSoup
import requests
from fake_useragent import UserAgent
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
import pandas as pd
import numpy as np
import threading 
from concurrent.futures import ThreadPoolExecutor
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [35]:
lock = threading.Lock()

fight_details = []
new_fight_links_all = []
winner_names = []
fighter_detail_data = []

In [36]:
MAX_THREADS = 10 # change this to adjust the number of concurrent threads

ua = UserAgent()
chrome = ua.chrome

HEADER = {
    'User-Agent' : chrome
}

In [37]:
def create_session(): # Create a session with retry strategy
    """Create a requests session with retry strategy for handling network issues."""
    # This function sets up a session with a retry strategy to handle network issues. 
    
    session = requests.Session()
    retry_strat = Retry(
        backoff_factor=1,
        total=3,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=['GET']
    )
    adapter = HTTPAdapter(max_retries= retry_strat)
    session.mount('https://', adapter)
    session.mount('http://', adapter)
    return session

session = create_session()

# Scraping the event links

In [38]:
# OLD LOGIC - no longer works

# ufc_link = "http://ufcstats.com/statistics/events/completed?page=all"

# respone = session.get(ufc_link)

# text = respone.text
# soup = BeautifulSoup(text, 'lxml')

# event_links_soup = soup.find_all('a', class_ = 'b-link b-link_style_black')

# event_links = [link['href'] for link in event_links_soup] # Extracting href attributes from the links

# print(len(event_links), "events found")

In [39]:
import subprocess
import sys
from pathlib import Path

backend_dir = Path.cwd()

script_path = backend_dir / "get_event_links.py"
links_path = backend_dir / "data" / "event_links.txt"

print("Getting UFCStats event links...")

result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("get_event_links.py failed")

with open(links_path, "r") as f:
    event_links = [line.strip() for line in f if line.strip()]

print(f"{len(event_links)} events loaded.")

Getting UFCStats event links...
786 events found
http://ufcstats.com/event-details/9d61d8cb1c354867
http://ufcstats.com/event-details/a0a69dc9914ef6e1
http://ufcstats.com/event-details/b96619b3acd7d9da
Saved event links to data\event_links.txt

786 events loaded.


# Scraping the event info

In [40]:
# def get_event_data(item): # Function to scrape event data
#     """Scrape event data from the given link."""
#     idx, link = item
#     link = link.strip()
#     response = session.get(link, headers=HEADER, timeout= 15)
#     response.raise_for_status()
#     if (response.status_code == 200):
#         soup = BeautifulSoup(response.text, 'lxml')
                
#         event_id = link[-16:]
#         date_loc_list = soup.find_all('li', 'b-list__box-list-item')
#         date = date_loc_list[0].text.replace("Date:", "").strip()
#         location = date_loc_list[1].text.replace("Location:", "").strip()
#         fight_links = soup.find_all('tr', class_ = 'b-fight-details__table-row b-fight-details__table-row__hover js-fight-details-click')
#         for i in fight_links:
#             winner_name = None
#             winner_id = None
#             w_l_d = i.find('i', class_ = "b-flag__text").text
#             fight_id = i['data-link'][-16:]
#             # print(w_l_d)
#             if w_l_d == "win":
#                 players = i.find('td', class_ = "b-fight-details__table-col l-page_align_left")
#                 players = players.find_all('a', class_= "b-link b-link_style_black")
#                 winner_name = players[0].text.strip()
#                 winner_id = players[0]['href'][-16:]
#             # Making the data
#             data_dic = {
#                 "event_id" : event_id,
#                 "fight_id" : fight_id,
#                 "date" : date,
#                 "location" : location,
#                 "winner" : winner_name,
#                 "winner_id" : winner_id
#             }
#             new_fight_links_all.append(i['data-link'])
#             winner_names.append(data_dic)
#         # print(f"Scrapped : {link}, {idx+1} / {len(event_links)}")
#         idx += 1
#     else:
#         print("Could'nt retrive the link." + str(response.status_code))

# with ThreadPoolExecutor(max_workers= MAX_THREADS) as executor:
#     results = [executor.submit(get_event_data, item) for item in enumerate(event_links)]
#     for r in results:
#         r.result()

# df_winner = pd.DataFrame(data=winner_names)
# df_winner.to_csv("event_details.csv", index = False)
# print(f"Successfully scrapped {len(df_winner)} event data.")
# df_winner

In [41]:
import subprocess
import sys
from pathlib import Path
import pandas as pd

backend_dir = Path.cwd()

script_path = backend_dir / "scrape_events.py"
event_details_path = backend_dir / "data" / "event_details.csv"
fight_links_path = backend_dir / "data" / "fight_links.txt"

print("Scraping UFCStats event data...")

result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("scrape_events.py failed")

df_winner = pd.read_csv(event_details_path)

with open(fight_links_path, "r") as f:
    new_fight_links_all = [
        line.strip()
        for line in f
        if line.strip()
    ]

df_event = pd.read_csv(backend_dir / "data" / "event_details.csv")
print(f"Event/fight records loaded: {len(df_winner)}")
print(f"Fight links loaded: {len(new_fight_links_all)}")

Scraping UFCStats event data...
Loaded 786 event links
Found 8758 existing fight records across previously-scraped events
Found 8832 existing fight links
8 events remaining to scrape

Scraping 8 events with 6 concurrent browser workers...
FAILED [6] http://ufcstats.com/event-details/f354c50b8d63d9b3
Error: Page.content: Unable to retrieve content because the page is navigating and changing the content.
FINAL saved: 8848 fight records, 8858 fight links

--------------------------------
SCRAPING COMPLETE
--------------------------------
Events processed this run: 8
New fight records: 90
New fight links: 90
Saved to: data\event_details.csv and data\fight_links.txt
--------------------------------

Event/fight records loaded: 8848
Fight links loaded: 8858


# Scraping the fight info

In [42]:
# def get_fight_data(item): # Function to scrape fight data
#     """Scrape fight data from the given link."""
#     idx, link = item
#     link = link.strip()
#     try:
#         response = session.get(link, headers=HEADER, timeout=15)
#         response.raise_for_status() 
        
#         soup = BeautifulSoup(response.text, 'lxml')
        
#         # event name
#         event_name = soup.find('a', class_ = "b-link").text.strip()
#         # event id
#         event_id = soup.find('a', class_ = "b-link")['href'][-16:]
#         # fight id
#         fight_id = link[-16:]
        
#         # fighter names
#         fighter_nams = soup.find_all('a', class_ = 'b-link b-fight-details__person-link')
#         r_name = fighter_nams[0].text.strip()
#         b_name = fighter_nams[1].text.strip()
        
#         # fighter ids
#         r_id = fighter_nams[0]['href'].strip()[-16:]
#         b_id = fighter_nams[1]['href'].strip()[-16:]
        
#         # title fight & division
#         division_info = soup.find('i', class_= 'b-fight-details__fight-title').text.lower()
#         is_title_fight = 0
#         if 'title' in division_info:
#             is_title_fight = 1
#         division_info = division_info.replace('ufc', "")
#         division_info = division_info.replace("title", "")
#         division_info = division_info.replace("bout", "").strip()
        
#         # method
#         method = soup.find('i', style = 'font-style: normal').text.strip()
        
        
#         p_tag_with_fight_detail = soup.find('p', class_ = "b-fight-details__text")
#         fight_details_list = p_tag_with_fight_detail.find_all('i', class_ = 'b-fight-details__text-item')
#         # finish-round
#         finish_round = int(fight_details_list[0].text.lower().replace("round:", "").strip())
#         # match-time
#         match_timestamp = fight_details_list[1].text.lower().replace("time:", "").strip()
#         match_timestamp_splited = match_timestamp.split(":")
#         match_time_sec = int(match_timestamp_splited[0]) * 60 + int(match_timestamp_splited[-1])
#         # total-round
#         total_rounds = fight_details_list[2].text.lower().replace("time format:", "").strip()
#         if total_rounds == "No Time Limit".lower():
#             total_rounds = None
#         else :
#             total_rounds = int(total_rounds[0])
#         # referee
#         referee = fight_details_list[3].text.replace("Referee:", "").strip()
        
        
#         # totals, SIG. STR.
#         tables = soup.find_all('table', style = "width: 745px")
        
#         # TOTALS TABLE
#         if len(tables) > 0:
#             table1 = tables[0]
#             td_1_list = table1.find_all('td', class_ = 'b-fight-details__table-col')
#             # KD
#             kd_players = td_1_list[1].text.split()
#             r_kd, b_kd = int(kd_players[0]), int(kd_players[1])
#             # sig. str.
#             sig_str_players = td_1_list[2].text.split() 
#             r_sig_str_landed = int(sig_str_players[0])
#             r_sig_str_atmpted = int(sig_str_players[2])
#             b_sig_str_landed = int(sig_str_players[3])
#             b_sig_str_atmpted = int(sig_str_players[5])
#             # sig_str_acc
#             sig_str_acc = td_1_list[3].text.split() 
#             r_sig_str_acc = int(sig_str_acc[0].replace("%", "")) if sig_str_acc[0] != "---" else None
#             b_sig_str_acc = int(sig_str_acc[1].replace("%", "")) if sig_str_acc[1] != "---" else None
#             # total-str
#             total_str = td_1_list[4].text.split() 
#             r_total_str_landed = int(total_str[0])
#             r_total_str_atmpted = int(total_str[2])
#             b_total_str_landed = int(total_str[3])
#             b_total_str_atmpted = int(total_str[5])
#             # total-str-acc
#             r_total_str_acc, b_total_str_acc = None, None
#             try:
#                 r_total_str_acc = int(round(r_total_str_landed / r_total_str_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_total_str_acc = int(round(b_total_str_landed / b_total_str_atmpted, 2) * 100)
#             except:
#                 pass
#             # TD
#             td_players = td_1_list[5].text.split() 
#             r_td_landed = int(td_players[0])
#             r_td_atmpted = int(td_players[2])
#             b_td_landed = int(td_players[3])
#             b_td_atmpted = int(td_players[5])
#             # td_acc
#             td_acc = td_1_list[6].text.split() 
#             r_td_acc = int(td_acc[0].replace("%", "")) if td_acc[0] != "---" else None
#             b_td_acc = int(td_acc[1].replace("%", "")) if td_acc[1] != "---" else None
#             # sub. att
#             sub_att = td_1_list[7].text.split()
#             r_sub_att, b_sub_att = int(sub_att[0]), int(sub_att[1])
#             # rev
#             rev = td_1_list[8].text.split()
#             r_rev, b_rev = int(rev[0]), int(rev[1])
#             # Ctrl
#             ctrl = td_1_list[9].text.split()
#             r_ctrl = ctrl[0].split(":")
#             r_ctrl = int(r_ctrl[0]) * 60 + int(r_ctrl[1]) if r_ctrl[0] != '--' else None
#             b_ctrl = ctrl[1].split(":")
#             b_ctrl = int(b_ctrl[0]) * 60 + int(b_ctrl[1]) if b_ctrl[0] != '--' else None
            
#             # SIG. STR. TABLE
#             table2 = tables[1]
#             td_2_list = table2.find_all('td', class_ = 'b-fight-details__table-col')
            
#             # HEAD
#             head_list = td_2_list[3].text.split() 
#             r_head_landed = int(head_list[0])
#             r_head_atmpted = int(head_list[2])
#             b_head_landed = int(head_list[3])
#             b_head_atmpted = int(head_list[5])
#             # HEAD
#             r_head_acc, b_head_acc = None, None
#             try:
#                 r_head_acc = int(round(r_head_landed / r_head_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_head_acc = int(round(b_head_landed / b_head_atmpted, 2) * 100)
#             except:
#                 pass
            
#             # BODY
#             body_list = td_2_list[4].text.split() 
#             r_body_landed = int(body_list[0])
#             r_body_atmpted = int(body_list[2])
#             b_body_landed = int(body_list[3])
#             b_body_atmpted = int(body_list[5])
#             # BODY ACC
#             r_body_acc, b_body_acc = None, None
#             try:
#                 r_body_acc = int(round(r_body_landed / r_body_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_body_acc = int(round(b_body_landed / b_body_atmpted, 2) * 100)
#             except:
#                 pass
            
#             # LEG
#             leg_list = td_2_list[5].text.split() 
#             r_leg_landed = int(leg_list[0])
#             r_leg_atmpted = int(leg_list[2])
#             b_leg_landed = int(leg_list[3])
#             b_leg_atmpted = int(leg_list[5])
#             # LEG ACC
#             r_leg_acc, b_leg_acc = None, None
#             try:
#                 r_leg_acc = int(round(r_leg_landed / r_leg_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_leg_acc = int(round(b_leg_landed / b_leg_atmpted, 2) * 100)
#             except:
#                 pass
            
#             # DISTANCE
#             dist_list = td_2_list[6].text.split() 
#             r_dist_landed = int(dist_list[0])
#             r_dist_atmpted = int(dist_list[2])
#             b_dist_landed = int(dist_list[3])
#             b_dist_atmpted = int(dist_list[5])
#             # DIST ACC
#             r_dist_acc, b_dist_acc = None, None
#             try:
#                 r_dist_acc = int(round(r_dist_landed / r_dist_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_dist_acc = int(round(b_dist_landed / b_dist_atmpted, 2) * 100)
#             except:
#                 pass
            
#             # CLINCH
#             clinch_list = td_2_list[7].text.split() 
#             r_clinch_landed = int(clinch_list[0])
#             r_clinch_atmpted = int(clinch_list[2])
#             b_clinch_landed = int(clinch_list[3])
#             b_clinch_atmpted = int(clinch_list[5])
#             # CLINCH ACC
#             r_clinch_acc, b_clinch_acc = None, None
#             try:
#                 r_clinch_acc = int(round(r_clinch_landed / r_clinch_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_clinch_acc = int(round(b_clinch_landed / b_clinch_atmpted, 2) * 100)
#             except:
#                 pass
            
#             # Ground
#             ground_list = td_2_list[8].text.split() 
#             r_ground_landed = int(ground_list[0])
#             r_ground_atmpted = int(ground_list[2])
#             b_ground_landed = int(ground_list[3])
#             b_ground_atmpted = int(ground_list[5])
#             # Ground ACC
#             r_ground_acc, b_ground_acc = None, None
#             try:
#                 r_ground_acc = int(round(r_ground_landed / r_ground_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_ground_acc = int(round(b_ground_landed / b_ground_atmpted, 2) * 100)
#             except:
#                 pass
#         else:
#             r_kd,b_kd = None, None
#             r_sig_str_landed,b_sig_str_landed = None, None
#             r_sig_str_atmpted,b_sig_str_atmpted = None, None
#             r_sig_str_acc,b_sig_str_acc = None, None
#             r_total_str_landed,b_total_str_landed = None, None
#             r_total_str_atmpted,b_total_str_atmpted = None, None
#             r_total_str_acc,b_total_str_acc = None, None
#             r_td_landed,b_td_landed= None, None
#             r_td_atmpted,b_td_atmpted = None, None
#             r_td_acc,b_td_acc= None, None
#             r_sub_att,b_sub_att= None, None
#             r_ctrl,b_ctrl= None, None
            
#             r_head_landed , b_head_landed = None, None
#             r_head_atmpted , b_head_atmpted = None, None
#             r_head_acc , b_head_acc = None, None
#             r_body_landed , b_body_landed = None, None
#             r_body_atmpted , b_body_atmpted = None, None
#             r_body_acc , b_body_acc = None, None
#             r_leg_landed , b_leg_landed = None, None
#             r_leg_atmpted , b_leg_atmpted = None, None
#             r_leg_acc , b_leg_acc = None, None
#             r_dist_landed , b_dist_landed = None, None
#             r_dist_atmpted , b_dist_atmpted = None, None
#             r_dist_acc , b_dist_acc = None, None
#             r_clinch_landed , b_clinch_landed = None, None
#             r_clinch_atmpted , b_clinch_atmpted= None, None
#             r_clinch_acc , b_clinch_acc = None, None
#             r_ground_landed , b_ground_landed = None, None
#             r_ground_atmpted , b_ground_atmpted = None, None
#             r_ground_acc , b_ground_acc = None, None
#             r_landed_head_per , b_landed_head_per = None, None
#             r_landed_body_per , b_landed_body_per= None, None
#             r_landed_leg_per , b_landed_leg_per = None, None
#             r_landed_dist_per , b_landed_dist_per = None, None
#             r_landed_clinch_per , b_landed_clinch_per = None, None
#             r_landed_ground_per , b_landed_ground_per = None, None
        
#         # LANDED-head&dist
#         try:
#             r_landed_head_and_dist_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_red b-fight-details__charts-num_pos_left js-red")
#             r_landed_head_per = int(r_landed_head_and_dist_list[0].text.strip().replace("%", ""))
#             r_landed_dist_per = int(r_landed_head_and_dist_list[1].text.strip().replace("%", ""))
#             b_landed_head_and_dist_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_blue b-fight-details__charts-num_pos_right js-blue")
#             b_landed_head_per = int(b_landed_head_and_dist_list[0].text.strip().replace("%", ""))
#             b_landed_dist_per = int(b_landed_head_and_dist_list[1].text.strip().replace("%", ""))
#         except:
#             r_landed_head_per, r_landed_dist_per = None, None
#             b_landed_head_per, b_landed_dist_per = None, None
#         # LANDED-Body&Clinch
#         try:
#             r_landed_body_and_clinch_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_dark-red b-fight-details__charts-num_pos_left js-red")
#             r_landed_body_per = int(r_landed_body_and_clinch_list[0].text.strip().replace("%", ""))
#             r_landed_clinch_per = int(r_landed_body_and_clinch_list[1].text.strip().replace("%", ""))
#             b_landed_body_and_clinch_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_dark-blue b-fight-details__charts-num_pos_right js-blue")
#             b_landed_body_per = int(b_landed_body_and_clinch_list[0].text.strip().replace("%", ""))
#             b_landed_clinch_per = int(b_landed_body_and_clinch_list[1].text.strip().replace("%", ""))
#         except:
#             r_landed_body_per, r_landed_clinch_per = None, None
#             b_landed_body_per, b_landed_clinch_per = None, None
            
#         # LANDED-leg&ground
#         try:
#             r_landed_leg_and_ground_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_light-red b-fight-details__charts-num_pos_left js-red")
#             r_landed_leg_per = int(r_landed_leg_and_ground_list[0].text.strip().replace("%", ""))
#             r_landed_ground_per = int(r_landed_leg_and_ground_list[1].text.strip().replace("%", ""))
#             b_landed_leg_and_ground_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_light-blue b-fight-details__charts-num_pos_right js-blue")
#             b_landed_leg_per = int(b_landed_leg_and_ground_list[0].text.strip().replace("%", ""))
#             b_landed_ground_per = int(b_landed_leg_and_ground_list[1].text.strip().replace("%", ""))
#         except:
#             # pass
#             r_landed_leg_per, r_landed_ground_per = None, None
#             b_landed_leg_per, b_landed_ground_per = None, None
            
#         # MAKING THE DATA
#         data_dic = {
#             "event_name" : event_name,
#             "event_id" : event_id,
#             "fight_id" : fight_id,
#             "r_name" : r_name,
#             "r_id" : r_id,
#             "b_name" : b_name,
#             "b_id" : b_id,
#             "division" : division_info,
#             "title_fight" : is_title_fight,
#             "method" : method,
#             "finish_round" : finish_round,
#             "match_time_sec" : match_time_sec,
#             "total_rounds" : total_rounds,
#             "referee" : referee,
#             "r_kd" : r_kd,
#             "r_sig_str_landed" : r_sig_str_landed,
#             "r_sig_str_atmpted" : r_sig_str_atmpted,
#             "r_sig_str_acc" : r_sig_str_acc,
#             "r_total_str_landed" : r_total_str_landed,
#             "r_total_str_atmpted" : r_total_str_atmpted,
#             "r_total_str_acc" : r_total_str_acc,
#             "r_td_landed" : r_td_landed,
#             "r_td_atmpted" : r_td_atmpted,
#             "r_td_acc" : r_td_acc,
#             "r_sub_att" : r_sub_att,
#             "r_ctrl" : r_ctrl,
#             "r_head_landed" : r_head_landed,
#             "r_head_atmpted" : r_head_atmpted,
#             "r_head_acc" : r_head_acc,
#             "r_body_landed" : r_body_landed,
#             "r_body_atmpted" : r_body_atmpted,
#             "r_body_acc" : r_body_acc,
#             "r_leg_landed" : r_leg_landed,
#             "r_leg_atmpted" : r_leg_atmpted,
#             "r_leg_acc" : r_leg_acc,
#             "r_dist_landed" : r_dist_landed,
#             "r_dist_atmpted" : r_dist_atmpted,
#             "r_dist_acc" : r_dist_acc,
#             "r_clinch_landed" : r_clinch_landed,
#             "r_clinch_atmpted" : r_clinch_atmpted,
#             "r_clinch_acc" : r_clinch_acc,
#             "r_ground_landed" : r_ground_landed,
#             "r_ground_atmpted" : r_ground_atmpted,
#             "r_ground_acc" : r_ground_acc,
#             "r_landed_head_per" : r_landed_head_per,
#             "r_landed_body_per" : r_landed_body_per,
#             "r_landed_leg_per" : r_landed_leg_per,
#             "r_landed_dist_per" : r_landed_dist_per,
#             "r_landed_clinch_per" : r_landed_clinch_per,
#             "r_landed_ground_per" : r_landed_ground_per,
#             "b_kd" : b_kd,
#             "b_sig_str_landed" : b_sig_str_landed,
#             "b_sig_str_atmpted" : b_sig_str_atmpted,
#             "b_sig_str_acc" : b_sig_str_acc,
#             "b_total_str_landed" : b_total_str_landed,
#             "b_total_str_atmpted" : b_total_str_atmpted,
#             "b_total_str_acc" : b_total_str_acc,
#             "b_td_landed" : b_td_landed,
#             "b_td_atmpted" : b_td_atmpted,
#             "b_td_acc" : b_td_acc,
#             "b_sub_att" : b_sub_att,
#             "b_ctrl" : b_ctrl,
#             "b_head_landed" : b_head_landed,
#             "b_head_atmpted" : b_head_atmpted,
#             "b_head_acc" : b_head_acc,
#             "b_body_landed" : b_body_landed,
#             "b_body_atmpted" : b_body_atmpted,
#             "b_body_acc" : b_body_acc,
#             "b_leg_landed" : b_leg_landed,
#             "b_leg_atmpted" : b_leg_atmpted,
#             "b_leg_acc" : b_leg_acc,
#             "b_dist_landed" : b_dist_landed,
#             "b_dist_atmpted" : b_dist_atmpted,
#             "b_dist_acc" : b_dist_acc,
#             "b_clinch_landed" : b_clinch_landed,
#             "b_clinch_atmpted" : b_clinch_atmpted,
#             "b_clinch_acc" : b_clinch_acc,
#             "b_ground_landed" : b_ground_landed,
#             "b_ground_atmpted" : b_ground_atmpted,
#             "b_ground_acc" : b_ground_acc,
#             "b_landed_head_per" : b_landed_head_per,
#             "b_landed_body_per" : b_landed_body_per,
#             "b_landed_leg_per" : b_landed_leg_per,
#             "b_landed_dist_per" : b_landed_dist_per,
#             "b_landed_clinch_per" : b_landed_clinch_per,
#             "b_landed_ground_per" : b_landed_ground_per
#         }
#         with lock:
#             fight_details.append(data_dic)
#             # print(f"Scraped {idx+1}/{len(new_fight_links_all)}: {link}")
#             idx += 1
#     except Exception as e:
#         print(f"FAILED [{idx}] {link}")
#         print(f"{type(e).__name__}: {e}")
#         return

# with ThreadPoolExecutor(max_workers= MAX_THREADS) as executor:
#     results = [executor.submit(get_fight_data, item) for item in enumerate(new_fight_links_all)]
#     for r in results:
#         r.result()
        
# print(f"Successfully scraped all fight data. Scrapped data {len(fight_details)}")

In [43]:
# df_fight = pd.DataFrame(data=fight_details)
# df_fight.to_csv("fight_details.csv", index = False)
# df_fight

In [44]:
process = subprocess.Popen(
    [sys.executable, "scrape_fights.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1
)

for line in process.stdout:
    print(line, end="")

process.wait()

print("Finished with code:", process.returncode)

if process.returncode != 0:
    raise RuntimeError("scrape_fights.py failed")

df_fight = pd.read_csv(backend_dir / "data" / "fight_details.csv")
print(f"Fight records loaded: {len(df_fight)}")

Loaded 8858 fight links
Found 8806 existing fights — will skip these and append new ones
54 fights remaining to scrape

Scraping 54 fights with 6 concurrent browser workers...

Fights:   0%|          | 0/54 [00:00<?, ?it/s]FAILED [224] - Title: Loading… HTML: 2994

Fights:   2%|▏         | 1/54 [00:02<02:01,  2.30s/it]FAILED [512] http://ufcstats.com/fight-details/0e60ab32f3d03929 — Playwright could not load page: Error

Fights:   4%|▎         | 2/54 [00:03<01:20,  1.55s/it]FAILED [31] - Title: Loading… HTML: 2994FAILED [47] - Title: Loading… HTML: 2994

FAILED [448] - Title: Loading… HTML: 2994
Fights:   6%|▌         | 3/54 [00:03<00:53,  1.06s/it]
FAILED [55] - Title: Loading… HTML: 2994FAILED [261] - Title: Loading… HTML: 2994


Fights:  11%|█         | 6/54 [00:04<00:20,  2.35it/s]FAILED [628] - Title: None HTML: 66

Fights:  15%|█▍        | 8/54 [00:04<00:13,  3.46it/s]FAILED [645] - Title: Loading… HTML: 2994

Fights:  17%|█▋        | 9/54 [00:06<00:32,  1.39it/s]FAILED [714] - T

# Scraping the fighter info

In [45]:
process = subprocess.Popen(
    [sys.executable, "scrape_fighter_details.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1
)

for line in process.stdout:
    print(line, end="")

process.wait()

print("Finished with code:", process.returncode)

if process.returncode != 0:
    raise RuntimeError("scrape_fighter_details.py failed")

df_fighter = pd.read_csv(backend_dir / "data" / "fighter_details.csv")
print(f"Fighter records loaded: {len(df_fighter)}")

Found 2682 existing fighter records
2735 unique fighter IDs found in fight data
Found 50 fighters flagged for refresh (had a new fight this cycle)
Never scraped before: 53
Needing refresh from new fights: 50
92 fighters remaining to scrape

Scraping 92 fighters with 6 concurrent browser workers...

Fighters:   0%|          | 0/92 [00:00<?, ?it/s]FAILED http://ufcstats.com/fighter-details/008ea710276c9606 — page did not contain expected fighter title (possibly blocked or not loaded)

Fighters:   1%|          | 1/92 [00:05<08:12,  5.42s/it]FAILED http://ufcstats.com/fighter-details/0052de90691d4a93 — page did not contain expected fighter title (possibly blocked or not loaded)
FAILED http://ufcstats.com/fighter-details/009c4420727149ea — Error: Page.goto: net::ERR_ABORTED at http://ufcstats.com/fighter-details/009c4420727149ea
Call log:
  - navigating to "http://ufcstats.com/fighter-details/009c4420727149ea", waiting until "domcontentloaded"


Fighters:   3%|▎         | 3/92 [00:05<02:19,

In [46]:
df_fighter

,id,name,nick_name,wins,losses,draws,height,weight,reach,stance,dob,splm,str_acc,sapm,str_def,td_avg,td_avg_acc,td_def,sub_avg
0,0112352cb32f5026,Denis Tiuliulin,NaN,10,10,0,185.42,83.91,195.58,Orthodox,"May 17, 1988",3.61,41,5.23,38,0.96,42,72,0.0
1,013da757877044a2,Joe Brammer,The South Side Strangler,7,3,1,172.72,70.31,NaN,Orthodox,"Aug 23, 1983",2.15,31,2.80,56,0.81,50,50,0.0
2,00e11b5c8b7bfeeb,Luke Rockhold,NaN,16,6,0,190.50,83.91,195.58,Southpaw,"Oct 17, 1984",4.10,49,2.68,53,0.70,29,65,1.0
3,01641ba5df0c69b0,Gabriel Bonfim,Marretinha,20,1,0,185.42,77.11,182.88,Orthodox,"Aug 20, 1997",4.76,47,3.61,64,2.57,53,78,1.0
4,01d2ed8c502e3828,Caros Fodor,The Future,11,6,0,175.26,70.31,187.96,Orthodox,"Jan 07, 1984",2.76,54,2.83,54,2.10,25,50,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2686,d3df1add9d9a7efb,Derrick Lewis,The Black Beast,29,14,0,190.50,117.93,200.66,Orthodox,"Feb 07, 1985",2.41,48,2.79,38,0.53,25,51,0.0
2687,cffb87059e645bd1,Denise Gomes,Dee,13,3,0,157.48,52.16,160.02,Orthodox,"Dec 30, 1999",4.43,49,3.16,51,1.45,34,66,0.7
2688,5870c541798124ae,Manoel Sousa,Manumito,15,1,0,175.26,70.31,177.80,Orthodox,"Jun 04, 1997",4.27,48,3.48,55,0.36,20,73,1.1
2689,e1f8beeb6d2d871a,Namsrai Batbayar,Steppe Warrior,10,2,0,162.56,56.70,172.72,Orthodox,"Dec 29, 2000",4.44,45,3.42,48,2.85,30,100,0.9


# Building the final data, by merging the tables 

In [64]:
# ============================================================
# BUILD UFC.csv FOR EXISTING MODEL
# Produces the legacy UFC.csv schema expected by the model
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")

# ============================================================
# LOAD RAW DATA
# ============================================================

df_event = pd.read_csv(DATA_DIR / "event_details.csv")
df_fight = pd.read_csv(DATA_DIR / "fight_details.csv")
df_fighter = pd.read_csv(DATA_DIR / "fighter_details.csv")

# Clean column names
df_event.columns = df_event.columns.str.strip()
df_fight.columns = df_fight.columns.str.strip()
df_fighter.columns = df_fighter.columns.str.strip()

print(f"Events loaded:   {len(df_event):,}")
print(f"Fights loaded:   {len(df_fight):,}")
print(f"Fighters loaded: {len(df_fighter):,}")


# ============================================================
# FIGHTER DATA
# ============================================================

df_fighter = df_fighter.rename(columns={
    "id": "fighter_id",
    "name": "fighter_name"
})

df_fighter["fighter_id"] = (
    df_fighter["fighter_id"]
    .astype(str)
    .str.strip()
)

df_fighter["fighter_name"] = (
    df_fighter["fighter_name"]
    .astype(str)
    .str.strip()
)

df_fighter = df_fighter.drop_duplicates(
    subset=["fighter_id"],
    keep="first"
).copy()


# ============================================================
# CLEAN EVENT DATA
# ============================================================

for col in ["event_id", "fight_id", "winner"]:
    if col in df_event.columns:
        df_event[col] = (
            df_event[col]
            .astype(str)
            .str.strip()
        )

df_event["date"] = pd.to_datetime(
    df_event["date"],
    errors="coerce",
    format="mixed"
)


# ============================================================
# CLEAN FIGHT DATA
# ============================================================

for col in [
    "event_id",
    "fight_id",
    "r_id",
    "b_id",
    "r_name",
    "b_name"
]:
    if col in df_fight.columns:
        df_fight[col] = (
            df_fight[col]
            .astype(str)
            .str.strip()
        )


# ============================================================
# ADD EVENT INFORMATION
# ============================================================

event_info = df_event[
    ["fight_id", "event_id", "date", "winner"]
].drop_duplicates(
    subset=["fight_id"],
    keep="first"
)

df = df_fight.merge(
    event_info,
    on="fight_id",
    how="left",
    suffixes=("", "_event")
)

if "event_id_event" in df.columns:
    if "event_id" in df.columns:
        df["event_id"] = df["event_id"].fillna(
            df["event_id_event"]
        )
    else:
        df["event_id"] = df["event_id_event"]

    df = df.drop(columns=["event_id_event"])

if "date_event" in df.columns:
    df["date"] = df["date_event"]
    df = df.drop(columns=["date_event"])

if "winner_event" in df.columns:
    df["winner"] = df["winner_event"]
    df = df.drop(columns=["winner_event"])


# ============================================================
# MERGE RED FIGHTER
# ============================================================

red = df_fighter.copy()

red = red.rename(columns={
    col: f"r_{col}_x"
    for col in red.columns
    if col not in ["fighter_id", "fighter_name"]
})

# Preserve fighter name as the legacy-style name field
red = red.rename(columns={
    "fighter_name": "fighter_name_r"
})

df = df.merge(
    red,
    left_on="r_id",
    right_on="fighter_id",
    how="left"
)

df = df.drop(
    columns=["fighter_id"],
    errors="ignore"
)


# ============================================================
# MERGE BLUE FIGHTER
# ============================================================

blue = df_fighter.copy()

blue = blue.rename(columns={
    col: f"b_{col}_y"
    for col in blue.columns
    if col not in ["fighter_id", "fighter_name"]
})

blue = blue.rename(columns={
    "fighter_name": "fighter_name_b"
})

df = df.merge(
    blue,
    left_on="b_id",
    right_on="fighter_id",
    how="left"
)

df = df.drop(
    columns=["fighter_id"],
    errors="ignore"
)


# ============================================================
# BASIC VALIDATION
# ============================================================

required_basic = [
    "date",
    "fight_id",
    "r_name",
    "b_name",
    "winner"
]

missing_basic = [
    col for col in required_basic
    if col not in df.columns
]

if missing_basic:
    print("\nMISSING BASIC COLUMNS:")
    print(missing_basic)
    print("\nAvailable columns:")
    print(df.columns.tolist())
    raise RuntimeError("Basic UFC columns are missing.")


# ============================================================
# CLEAN DATE
# ============================================================

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce",
    format="mixed"
)

df = df.dropna(
    subset=["date"]
).copy()


# ============================================================
# CLEAN WINNER
# ============================================================

df["winner"] = (
    df["winner"]
    .astype(str)
    .str.strip()
)


# ============================================================
# REMOVE DUPLICATE FIGHTS
# ============================================================

df = df.drop_duplicates(
    subset=["fight_id"],
    keep="first"
).copy()


# ============================================================
# RESTORE LEGACY UNSUFFIXED FIGHTER COLUMNS
#
# The old UFC.csv contained both:
#
#     r_wins_x
#     r_wins_y
#     r_wins
#
# The current raw data only naturally creates the _x/_y
# versions. The existing model expects the unsuffixed versions.
# ============================================================

legacy_pairs = {
    # RED
    "r_nick_name": "r_nick_name_x",
    "r_wins": "r_wins_x",
    "r_losses": "r_losses_x",
    "r_draws": "r_draws_x",
    "r_height": "r_height_x",
    "r_weight": "r_weight_x",
    "r_reach": "r_reach_x",
    "r_stance": "r_stance_x",
    "r_dob": "r_dob_x",
    "r_splm": "r_splm_x",
    "r_str_acc": "r_str_acc_x",
    "r_sapm": "r_sapm_x",
    "r_str_def": "r_str_def_x",
    "r_td_avg": "r_td_avg_x",
    "r_td_avg_acc": "r_td_avg_acc_x",
    "r_td_def": "r_td_def_x",
    "r_sub_avg": "r_sub_avg_x",

    # BLUE
    "b_nick_name": "b_nick_name_y",
    "b_wins": "b_wins_y",
    "b_losses": "b_losses_y",
    "b_draws": "b_draws_y",
    "b_height": "b_height_y",
    "b_weight": "b_weight_y",
    "b_reach": "b_reach_y",
    "b_stance": "b_stance_y",
    "b_dob": "b_dob_y",
    "b_splm": "b_splm_y",
    "b_str_acc": "b_str_acc_y",
    "b_sapm": "b_sapm_y",
    "b_str_def": "b_str_def_y",
    "b_td_avg": "b_td_avg_y",
    "b_td_avg_acc": "b_td_avg_acc_y",
    "b_td_def": "b_td_def_y",
    "b_sub_avg": "b_sub_avg_y",
}

for legacy_col, source_col in legacy_pairs.items():
    if legacy_col not in df.columns:
        if source_col in df.columns:
            df[legacy_col] = df[source_col]
        else:
            df[legacy_col] = np.nan


# ============================================================
# MAKE SURE LEGACY EVENT COLUMNS EXIST
# ============================================================

if "event_name" not in df.columns and "event_id" in df.columns:
    event_names = df_event[
        ["event_id", "event_name"]
    ].drop_duplicates("event_id")

    df = df.merge(
        event_names,
        on="event_id",
        how="left",
        suffixes=("", "_eventname")
    )

    if "event_name_eventname" in df.columns:
        if "event_name" not in df.columns:
            df["event_name"] = df["event_name_eventname"]

        df = df.drop(
            columns=["event_name_eventname"],
            errors="ignore"
        )


# ============================================================
# ADD LOCATION IF AVAILABLE
# ============================================================

if "location" not in df.columns:
    if "location" in df_event.columns:
        locations = df_event[
            ["event_id", "location"]
        ].drop_duplicates("event_id")

        df = df.merge(
            locations,
            on="event_id",
            how="left",
            suffixes=("", "_event")
        )

        if "location_event" in df.columns:
            df["location"] = df["location_event"]
            df = df.drop(columns=["location_event"])


# ============================================================
# ENSURE FIGHTER PROFILE COLUMNS EXIST
# ============================================================

# These are the columns the old model specifically uses.
required_profile = [
    "r_dob_x",
    "b_dob_y",
    "r_splm_x",
    "b_splm_y",
    "r_sapm_x",
    "b_sapm_y",
    "r_str_acc_x",
    "b_str_acc_y",
    "r_str_def_x",
    "b_str_def_y",
    "r_td_avg_x",
    "b_td_avg_y",
    "r_td_avg_acc_x",
    "b_td_avg_acc_y",
    "r_td_def_x",
    "b_td_def_y",
    "r_sub_avg_x",
    "b_sub_avg_y",
    "r_reach_x",
    "b_reach_y",
    "r_height_x",
    "b_height_y",
    "r_weight_x",
    "b_weight_y",
    "r_wins_x",
    "b_wins_y",
    "r_losses_x",
    "b_losses_y"
]

missing_profile = [
    col for col in required_profile
    if col not in df.columns
]

if missing_profile:
    print("\nMISSING FIGHTER PROFILE COLUMNS:")
    for col in missing_profile:
        print(col)

    raise RuntimeError(
        f"{len(missing_profile)} fighter profile columns are missing."
    )


# ============================================================
# LEGACY COLUMN ORDER
#
# This recreates the structure of the OLD UFC.csv.
# Columns that are not available in the raw data are created
# as NaN rather than changing the model schema.
# ============================================================

legacy_schema = [
    "event_id",
    "event_name",
    "date",
    "location",
    "fight_id",
    "division",
    "title_fight",
    "method",
    "finish_round",
    "match_time_sec",
    "total_rounds",
    "referee",

    # RED fight statistics
    "r_name",
    "r_id",
    "r_kd",
    "r_sig_str_landed",
    "r_sig_str_atmpted",
    "r_sig_str_acc",
    "r_total_str_landed",
    "r_total_str_atmpted",
    "r_total_str_acc",
    "r_td_landed",
    "r_td_atmpted",
    "r_td_acc",
    "r_sub_att",
    "r_ctrl",
    "r_head_landed",
    "r_head_atmpted",
    "r_head_acc",
    "r_body_landed",
    "r_body_atmpted",
    "r_body_acc",
    "r_leg_landed",
    "r_leg_atmpted",
    "r_leg_acc",
    "r_dist_landed",
    "r_dist_atmpted",
    "r_dist_acc",
    "r_clinch_landed",
    "r_clinch_atmpted",
    "r_clinch_acc",
    "r_ground_landed",
    "r_ground_atmpted",
    "r_ground_acc",
    "r_landed_head_per",
    "r_landed_body_per",
    "r_landed_leg_per",
    "r_landed_dist_per",
    "r_landed_clinch_per",
    "r_landed_ground_per",

    # RED fighter profile X
    "r_nick_name_x",
    "r_wins_x",
    "r_losses_x",
    "r_draws_x",
    "r_height_x",
    "r_weight_x",
    "r_reach_x",
    "r_stance_x",
    "r_dob_x",
    "r_splm_x",
    "r_str_acc_x",
    "r_sapm_x",
    "r_str_def_x",
    "r_td_avg_x",
    "r_td_avg_acc_x",
    "r_td_def_x",
    "r_sub_avg_x",

    # RED fighter profile Y
    "r_nick_name_y",
    "r_wins_y",
    "r_losses_y",
    "r_draws_y",
    "r_height_y",
    "r_weight_y",
    "r_reach_y",
    "r_stance_y",
    "r_dob_y",
    "r_splm_y",
    "r_str_acc_y",
    "r_sapm_y",
    "r_str_def_y",
    "r_td_avg_y",
    "r_td_avg_acc_y",
    "r_td_def_y",
    "r_sub_avg_y",

    # RED legacy unsuffixed
    "r_nick_name",
    "r_wins",
    "r_losses",
    "r_draws",
    "r_height",
    "r_weight",
    "r_reach",
    "r_stance",
    "r_dob",
    "r_splm",
    "r_str_acc",
    "r_sapm",
    "r_str_def",
    "r_td_avg",
    "r_td_avg_acc",
    "r_td_def",
    "r_sub_avg",

    # BLUE fight statistics
    "b_name",
    "b_id",
    "b_kd",
    "b_sig_str_landed",
    "b_sig_str_atmpted",
    "b_sig_str_acc",
    "b_total_str_landed",
    "b_total_str_atmpted",
    "b_total_str_acc",
    "b_td_landed",
    "b_td_atmpted",
    "b_td_acc",
    "b_sub_att",
    "b_ctrl",
    "b_head_landed",
    "b_head_atmpted",
    "b_head_acc",
    "b_body_landed",
    "b_body_atmpted",
    "b_body_acc",
    "b_leg_landed",
    "b_leg_atmpted",
    "b_leg_acc",
    "b_dist_landed",
    "b_dist_atmpted",
    "b_dist_acc",
    "b_clinch_landed",
    "b_clinch_atmpted",
    "b_clinch_acc",
    "b_ground_landed",
    "b_ground_atmpted",
    "b_ground_acc",
    "b_landed_head_per",
    "b_landed_body_per",
    "b_landed_leg_per",
    "b_landed_dist_per",
    "b_landed_clinch_per",
    "b_landed_ground_per",

    # BLUE fighter profile X
    "b_nick_name_x",
    "b_wins_x",
    "b_losses_x",
    "b_draws_x",
    "b_height_x",
    "b_weight_x",
    "b_reach_x",
    "b_stance_x",
    "b_dob_x",
    "b_splm_x",
    "b_str_acc_x",
    "b_sapm_x",
    "b_str_def_x",
    "b_td_avg_x",
    "b_td_avg_acc_x",
    "b_td_def_x",
    "b_sub_avg_x",

    # BLUE fighter profile Y
    "b_nick_name_y",
    "b_wins_y",
    "b_losses_y",
    "b_draws_y",
    "b_height_y",
    "b_weight_y",
    "b_reach_y",
    "b_stance_y",
    "b_dob_y",
    "b_splm_y",
    "b_str_acc_y",
    "b_sapm_y",
    "b_str_def_y",
    "b_td_avg_y",
    "b_td_avg_acc_y",
    "b_td_def_y",
    "b_sub_avg_y",

    # BLUE legacy unsuffixed
    "b_nick_name",
    "b_wins",
    "b_losses",
    "b_draws",
    "b_height",
    "b_weight",
    "b_reach",
    "b_stance",
    "b_dob",
    "b_splm",
    "b_str_acc",
    "b_sapm",
    "b_str_def",
    "b_td_avg",
    "b_td_avg_acc",
    "b_td_def",
    "b_sub_avg",

    "winner",
    "winner_id"
]


# ============================================================
# CREATE ANY MISSING LEGACY COLUMNS
# ============================================================

for col in legacy_schema:
    if col not in df.columns:
        df[col] = np.nan


# ============================================================
# WINNER ID
# ============================================================

if "winner_id" not in df.columns or df["winner_id"].isna().all():

    def get_winner_id(row):
        winner = str(row["winner"]).strip()

        if winner == str(row["r_name"]).strip():
            return row["r_id"]

        if winner == str(row["b_name"]).strip():
            return row["b_id"]

        return np.nan

    df["winner_id"] = df.apply(
        get_winner_id,
        axis=1
    )


# ============================================================
# FINAL COLUMN ORDER
# ============================================================

df = df[legacy_schema].copy()


# ============================================================
# SORT NEWEST → OLDEST
# ============================================================

df = (
    df
    .sort_values(
        by="date",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# SAVE
# ============================================================

output_path = DATA_DIR / "UFC.csv"

df.to_csv(
    output_path,
    index=False
)


# ============================================================
# VERIFY
# ============================================================

print()
print("=" * 70)
print("UFC.csv SUCCESSFULLY CREATED")
print("=" * 70)

print(f"Rows:    {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Output:  {output_path}")

print()
print("Date range:")
print(f"Newest: {df['date'].max().date()}")
print(f"Oldest: {df['date'].min().date()}")

print()
print("Winner values:")
print(df["winner"].value_counts().head())

print()
print("Schema verification:")

# These are the columns that caused the previous model error
critical_columns = [
    "r_reach",
    "b_reach",
    "r_height",
    "b_height",
    "r_wins",
    "b_wins",
    "r_losses",
    "b_losses",
    "winner"
]

for col in critical_columns:
    print(f"  {'OK' if col in df.columns else 'MISSING'}: {col}")

print()
print("Newest 5 fights:")
display(df.head())


Events loaded:   8,848
Fights loaded:   8,831
Fighters loaded: 2,691

UFC.csv SUCCESSFULLY CREATED
Rows:    8,820
Columns: 192
Output:  data\UFC.csv

Date range:
Newest: 2026-08-29
Oldest: 1994-03-11

Winner values:
winner
Jim Miller          28
Neil Magny          25
Charles Oliveira    25
Max Holloway        24
Andrei Arlovski     23
Name: count, dtype: int64

Schema verification:
  OK: r_reach
  OK: b_reach
  OK: r_height
  OK: b_height
  OK: r_wins
  OK: b_wins
  OK: r_losses
  OK: b_losses
  OK: winner

Newest 5 fights:


,event_id,event_name,date,location,fight_id,division,title_fight,method,finish_round,match_time_sec,...,b_splm,b_str_acc,b_sapm,b_str_def,b_td_avg,b_td_avg_acc,b_td_def,b_sub_avg,winner,winner_id
0,9d61d8cb1c354867,UFC Fight Night: Nurmagomedov vs. Song,2026-08-29,"Shanghai, China",ec5281a72f09d8ac,flyweight,0,Submission,1,254,...,3.87,42.0,5.27,50.0,0.00,0.0,72.0,0.0,Rei Tsuruya,2f43a3e82661fa99
1,9d61d8cb1c354867,UFC Fight Night: Nurmagomedov vs. Song,2026-08-29,"Shanghai, China",ab5922e10699e869,bantamweight,0,KO/TKO,2,53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hector Santiago,8e4902b4a8045070
2,9d61d8cb1c354867,UFC Fight Night: Nurmagomedov vs. Song,2026-08-29,"Shanghai, China",228fefc6923c40a5,bantamweight,0,KO/TKO,2,108,...,4.34,43.0,3.81,55.0,0.82,41.0,73.0,0.2,Song Yadong,efb96bf3e9ada36f
3,9d61d8cb1c354867,UFC Fight Night: Nurmagomedov vs. Song,2026-08-29,"Shanghai, China",48a5d9c4592d2f2a,flyweight,0,Decision - Unanimous,3,300,...,4.40,50.0,2.29,63.0,0.76,37.0,73.0,0.5,Sumudaerji,3cf18e01cb6cbde3
4,9d61d8cb1c354867,UFC Fight Night: Nurmagomedov vs. Song,2026-08-29,"Shanghai, China",80f7f5405d5ffca0,flyweight,0,Submission,3,183,...,3.94,56.0,2.66,58.0,1.11,60.0,75.0,0.9,Andre Lima,b1f21ce050035d58
